# Discovery: native graphs, explicit scores and reconnect

SDK 0.7 native causal example. Default execution is a synthetic HTTP wire fixture, not a discovery model or scientific benchmark. Explicit live execution requires the qualified ArrowFM causal task; it never substitutes another family. This notebook uses the frozen seed42 full128x4 observational chain/isolate input and retains every uploaded variable. The five-engine causal pack and public Ergodic co-release remain separate gates. No deletion or saved predictions occur.


In [ ]:
from support import notebook_causal_client, causal_table
from welt import CausalResult
import numpy as np

client = notebook_causal_client()
profile = next((m for m in client.models() if m["id"] == "arrow"), None)
task = next((p for p in (profile or {}).get("task_profiles", [])
             if p["task"] == "causal_discovery"), None)
if not task or not task.get("execution_available"):
    raise RuntimeError("Qualified ArrowFM discovery is unavailable; no substitution or upload.")
columns, observations = causal_table()
dataset = client.upload(columns=columns, rows=observations.tolist(), target=None,
                        name="Synthetic discovery SDK notebook")
job = client.submit_discover(dataset["id"], model="arrow", seed=42)
result = job.result(timeout=120)
assert isinstance(result, CausalResult)
assert result.job_id == job.id
assert result.variables == tuple(columns)
assert result.model_version == task["version"]
assert result.native_graph_type == "dag"


The native graph and edge scores are separate outputs. ArrowFM decodes its native DAG from the original skeleton and ordering, rather than thresholding the returned edge probabilities. Preserve names, endpoint marks, decoder and model/configuration provenance. Learned edges do not establish causal truth; scores are neither calibrated confidence nor graph posterior probabilities.


In [ ]:
scores = client.causal_scores(result.job_id)
values = np.asarray(scores["values"], dtype=float)
assert scores["result_id"] == result.id
assert scores["score_reference"] == result.score_reference
assert scores["variables"] == list(result.variables)
assert scores["score_axes"] == "source_row_target_column"
assert values.shape == (4, 4) and np.isfinite(values).all()
assert ((values >= 0) & (values <= 1)).all()
assert (values.diagonal() == 0).all()
reopened = client.causal_result(result.job_id)
assert reopened.to_dict() == result.to_dict()
assert all(edge["u"] in columns and edge["v"] in columns for edge in result.edge_marks)
operation_id = job.inspect()["operation_id"]
assert operation_id != job.id
events = [e for e in client.usage() if e["operation_id"] == operation_id]
assert events and all(e["model_version"] == result.model_version for e in events)
client.close()


Explicit local Ergodic conversion uses a separately installed approved co-release artifact. Welt does not download it. The actual compatible wheel is currently private; absence is shown below and is not counted as successful causal continuation. Conversion accepts only a declared native DAG without graph repair. MixedGraph stores node sets, so always supply the original order when inspecting adjacency. No effect estimate is claimed by this notebook.


In [ ]:
from welt import OptionalDependencyError

try:
    graph = result.to_ergodic()
except OptionalDependencyError:
    print("Local graph continuation is gated on the approved Ergodic co-release artifact.")
else:
    adjacency, order = graph.to_adjacency(order=list(result.variables))
    assert order == list(result.variables)
    assert adjacency.shape == (len(columns), len(columns))
    assert graph.nodes == frozenset(result.variables)


Primary graphs and scores persist until an explicit owned deletion under server quotas. They are separate from seven-day downloadable prediction results. These created synthetic datasets/jobs/results stay retained; reconnect with result.job_id and a fresh Client. Controlled transport validates client plumbing only; live native execution, tenant isolation, restart, resource admission and learned Ergodic identification/estimation need separate evidence.
